# mol3d_fullerene — Results Analysis (HP-Tuned)

Combined mol3d + fullerene experiment: models are trained on the pooled train set (mol3d + fullerene) and evaluated on the pooled test set. Unlike `analyze_results.ipynb` (hand-picked/default hyperparameters), every model here (except SchNet) was run with `--hp_file results/best_hp_<model>.json`, i.e. the hyperparameters found by that model's own `hp_tuning_<model>.py` sweep.

Models covered:
- **CT** (Cellular Transformer) — `feat_mode` in `{full, simple, coords}`
- **GNN** — GCN / GAT / GIN
- **CIN** — CIN / CIN++
- **SchNet** — does **not** have a hp-tuned run (`best_hp_schnet.json` doesn't exist, tuning never attempted), so it's included using its plain (non-tuned) result instead: `results_schnet.json`


In [ ]:
import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RESULTS_DIR = Path("results")

def load_result(path):
    with open(path) as f:
        return json.load(f)

def rerun_suffix(path):
    # results_GCN_hptuned.json -> 0, results_GCN_hptuned_1.json -> 1, ...
    m = re.search(r"_(\d+)\.json$", path.name)
    return int(m.group(1)) if m else 0

# group key: (model_family, feature_label) -- files like results_GCN_hptuned.json and
# results_GCN_hptuned_1.json would be reruns of the same config; keep only the newest
# (highest-suffix) rerun per group. Only "*_hptuned*.json" files are picked up here,
# so the plain (non-tuned) results from analyze_results.ipynb never leak in.
by_group = {}
for path in sorted(RESULTS_DIR.glob("results_*_hptuned*.json")):
    d = load_result(path)
    if "feat_mode" in d:
        family = "CT"
        feature = d["feat_mode"]
    else:
        family = d["model"]
        feature = "-"
    key = (family, feature)
    suffix = rerun_suffix(path)
    if key not in by_group or suffix > by_group[key]["suffix"]:
        by_group[key] = {"path": path, "suffix": suffix, "data": d}

# SchNet isn't hp-tuned (no results_schnet_hptuned.json / best_hp_schnet.json), so it's
# not picked up by the glob above -- add its plain (non-tuned) result manually.
schnet_path = RESULTS_DIR / "results_schnet.json"
if schnet_path.exists():
    by_group[("SchNet", "-")] = {"path": schnet_path, "suffix": 0, "data": load_result(schnet_path)}

records = []
for (family, feature), entry in by_group.items():
    d = entry["data"]
    records.append({
        "file": entry["path"].name, "family": family, "feature": feature,
        "test_mae":  [r["test_mae"]  for r in d["runs"]],
        "test_rmse": [r["test_rmse"] for r in d["runs"]],
        "test_r2":   [r["test_r2"]   for r in d["runs"]],
        "num_params": d.get("num_params"),
    })

print(f"Using {len(records)} result files (newest rerun per group):")
for r in records:
    print(f"  {r['file']:28s} family={r['family']:5s} feature={r['feature']:8s} n_runs={len(r['test_mae'])}")


## Summary table

In [ ]:
def label(r):
    return r["family"] if r["feature"] == "-" else f'{r["family"]} ({r["feature"]})'

summary = pd.DataFrame([
    {
        "label": label(r),
        "family": r["family"],
        "feature": r["feature"],
        "mean_mae": np.mean(r["test_mae"]),
        "std_mae": np.std(r["test_mae"]),
        "mean_rmse": np.mean(r["test_rmse"]),
        "mean_r2": np.mean(r["test_r2"]),
        "n_runs": len(r["test_mae"]),
        "num_params": r["num_params"],
    }
    for r in records
]).sort_values("mean_mae").reset_index(drop=True)

summary

## Test MAE by model / feature configuration

Lower is better. Error bars show ±1 std across the 3 seeded runs within each result file.

In [ ]:
# Validated categorical palette (light mode), assigned in fixed order per model family.
FAMILY_COLOR = {
    "CT":     "#2a78d6",  # blue
    "GCN":    "#1baf7a",  # aqua
    "GAT":    "#eda100",  # yellow
    "GIN":    "#008300",  # green
    "CIN":    "#8172b2",  # purple
    "CINpp":  "#937860",  # brown
    "SchNet": "#64B5CD",  # light blue
}
# CT has 3 feature variants sharing the CT hue, distinguished by shade.
CT_FEATURE_ALPHA = {"full": 1.0, "simple": 0.65, "coords": 0.4}

def bar_color(row):
    base = FAMILY_COLOR[row["family"]]
    if row["family"] == "CT":
        alpha = CT_FEATURE_ALPHA.get(row["feature"], 0.8)
        rgb = np.array([int(base[i:i+2], 16) for i in (1, 3, 5)]) / 255
        white = np.array([1.0, 1.0, 1.0])
        rgb = alpha * rgb + (1 - alpha) * white
        return tuple(rgb)
    return base

colors = [bar_color(row) for _, row in summary.iterrows()]

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(summary))
bars = ax.bar(x, summary["mean_mae"], yerr=summary["std_mae"], width=0.6,
               color=colors, edgecolor="none", capsize=3,
               error_kw={"ecolor": "#52514e", "elinewidth": 1})

for xi, row in zip(x, summary.itertuples()):
    ax.text(xi, row.mean_mae + row.std_mae + 0.01, f"{row.mean_mae:.3f}",
            ha="center", va="bottom", fontsize=9, color="#0b0b0b")

ax.set_xticks(x)
ax.set_xticklabels(summary["label"], rotation=30, ha="right")
ax.set_ylabel("Test MAE")
ax.set_title("mol3d_fullerene — Test MAE by model / feature configuration (hp-tuned)")
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.yaxis.grid(True, alpha=0.25)
ax.set_axisbelow(True)

legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLOR.values()]
ax.legend(legend_handles, FAMILY_COLOR.keys(), title="Model family",
          loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)

plt.tight_layout()
plt.show()


## Test MAE by source (mol3d vs. fullerene)

Same test set, split by which dataset each molecule came from (`source` field on the saved per-molecule predictions), to see whether error is concentrated in one subset.

In [ ]:
def source_mae(predictions, source):
    errs = [abs(p["pred"] - p["true"]) for p in predictions if p["source"] == source]
    return np.mean(errs)

records_by_source = []
for (family, feature), entry in by_group.items():
    d = entry["data"]
    mol3d_maes    = [source_mae(r["predictions"], "mol3d")     for r in d["runs"]]
    fullerene_maes = [source_mae(r["predictions"], "fullerene") for r in d["runs"]]
    records_by_source.append({
        "label": family if feature == "-" else f"{family} ({feature})",
        "family": family, "feature": feature,
        "mol3d_mae": mol3d_maes, "fullerene_mae": fullerene_maes,
    })

summary_source = pd.DataFrame([
    {
        "label": r["label"], "family": r["family"], "feature": r["feature"],
        "mean_mol3d_mae": np.mean(r["mol3d_mae"]), "std_mol3d_mae": np.std(r["mol3d_mae"]),
        "mean_fullerene_mae": np.mean(r["fullerene_mae"]), "std_fullerene_mae": np.std(r["fullerene_mae"]),
    }
    for r in records_by_source
])
# keep the same model order as the overall MAE chart, for easy side-by-side comparison
summary_source = summary_source.set_index("label").loc[summary["label"]].reset_index()
summary_source

In [ ]:
colors_source = [bar_color(row) for _, row in summary_source.iterrows()]
x = np.arange(len(summary_source))
ymax = max((summary_source["mean_mol3d_mae"] + summary_source["std_mol3d_mae"]).max(),
           (summary_source["mean_fullerene_mae"] + summary_source["std_fullerene_mae"]).max()) * 1.15

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), sharey=True)

for ax, col_mean, col_std, title in [
    (axes[0], "mean_mol3d_mae", "std_mol3d_mae", "mol3d test molecules"),
    (axes[1], "mean_fullerene_mae", "std_fullerene_mae", "fullerene test molecules"),
]:
    ax.bar(x, summary_source[col_mean], yerr=summary_source[col_std], width=0.6,
           color=colors_source, edgecolor="none", capsize=3,
           error_kw={"ecolor": "#52514e", "elinewidth": 1})
    for xi, mean, std in zip(x, summary_source[col_mean], summary_source[col_std]):
        ax.text(xi, mean + std + ymax * 0.015, f"{mean:.3f}",
                ha="center", va="bottom", fontsize=9, color="#0b0b0b")
    ax.set_xticks(x)
    ax.set_xticklabels(summary_source["label"], rotation=30, ha="right")
    ax.set_title(title)
    ax.set_ylim(0, ymax)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, alpha=0.25)
    ax.set_axisbelow(True)

axes[0].set_ylabel("Test MAE")
legend_handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in FAMILY_COLOR.values()]
axes[1].legend(legend_handles, FAMILY_COLOR.keys(), title="Model family",
               loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)

fig.suptitle("mol3d_fullerene — Test MAE by source dataset (hp-tuned)")
plt.tight_layout()
plt.show()

## Structural Complexity Analysis

Same style of analysis as `mol3d/dataset_histograms.ipynb`, extended to the combined mol3d + fullerene test set. Four structural-complexity metrics per molecule:
- **Graph diameter** — longest shortest-path distance between any two atoms (bonds)
- **Spatial diameter** — longest Euclidean distance between any two atoms (Å)
- **Cyclomatic number** — `|E| - |V| + 1`
- **Number of rings** (SSSR)

mol3d metrics come from the existing `../mol3d/dataset_per_molecule.npz` (indexed by the same global mol3d index used in the saved predictions). Fullerene metrics come from `fullerene_dataset_per_molecule.npz`, generated using the exact same metric functions (`../mol3d/analyze_dataset.py`) applied to the exact same RDKit mol objects the training scripts use (`fullerene_loader._iter_fullerene_mols`), indexed the same way as `full_split["test_idx"]`. (These metrics are a property of the molecule structure only, computed once — unaffected by which hyperparameters the models were run with.)

In [ ]:
from scipy.stats import rankdata, binned_statistic

MODEL_COLOR = {row["label"]: bar_color(row) for _, row in summary.iterrows()}

# Load once and extract plain arrays -- np.load on a savez_compressed file returns
# a lazy NpzFile that re-decompresses the WHOLE array on every __getitem__, so
# indexing it inside a per-molecule loop is catastrophically slow. Extract first.
mol3d_npz = np.load("../mol3d/dataset_per_molecule.npz")
mol3d_pos = {int(idx): i for i, idx in enumerate(mol3d_npz["index"])}
mol3d_gd, mol3d_sd = mol3d_npz["graph_diameter"], mol3d_npz["spatial_diameter"]
mol3d_cn, mol3d_nr = mol3d_npz["cyclomatic_number"], mol3d_npz["num_rings"]

fullerene_npz = np.load("fullerene_dataset_per_molecule.npz")
fullerene_pos = {int(idx): i for i, idx in enumerate(fullerene_npz["index"])}
fullerene_gd, fullerene_sd = fullerene_npz["graph_diameter"], fullerene_npz["spatial_diameter"]
fullerene_cn, fullerene_nr = fullerene_npz["cyclomatic_number"], fullerene_npz["num_rings"]

def get_metrics(source, index):
    """(graph_diameter, spatial_diameter, cyclomatic_number, num_rings) for one
    (source, index) prediction key, or None if not found."""
    if source == "mol3d":
        pos = mol3d_pos.get(index)
        if pos is None:
            return None
        return (float(mol3d_gd[pos]), float(mol3d_sd[pos]), float(mol3d_cn[pos]), float(mol3d_nr[pos]))
    pos = fullerene_pos.get(index)
    if pos is None:
        return None
    return (float(fullerene_gd[pos]), float(fullerene_sd[pos]), float(fullerene_cn[pos]), float(fullerene_nr[pos]))

def per_molecule_mae(d):
    """(source, index) -> mean absolute error across the 3 seeded runs."""
    err_by_key = {}
    for run in d["runs"]:
        for p in run["predictions"]:
            err_by_key.setdefault((p["source"], p["index"]), []).append(abs(p["pred"] - p["true"]))
    return {k: float(np.mean(v)) for k, v in err_by_key.items()}

model_errors, err_maps = {}, {}
for _, row in summary.iterrows():
    d = by_group[(row["family"], row["feature"])]["data"]
    mae_by_key = per_molecule_mae(d)
    err_maps[row["label"]] = mae_by_key

    gd_l, sd_l, cn_l, nr_l, err_l = [], [], [], [], []
    for (source, index), err in mae_by_key.items():
        m = get_metrics(source, index)
        if m is None:
            continue
        gd, sd, cn, nr = m
        gd_l.append(gd); sd_l.append(sd); cn_l.append(cn); nr_l.append(nr); err_l.append(err)
    model_errors[row["label"]] = {
        "gd": np.array(gd_l), "sd": np.array(sd_l),
        "cn": np.array(cn_l), "nr": np.array(nr_l), "err": np.array(err_l),
    }

model_labels = list(model_errors.keys())
for lbl in model_labels:
    print(f"{lbl:14s} n_molecules={len(model_errors[lbl]['err'])}")

### Absolute error per molecule vs structural complexity

Each dot = one test molecule (mol3d + fullerene combined), y = mean absolute error across the 3 seeded runs. Dashed line = linear fit.

In [ ]:
metrics = [('gd', 'Graph diameter (bonds)'), ('sd', 'Spatial diameter (Å)'),
           ('cn', 'Cyclomatic number'), ('nr', 'Number of rings (SSSR)')]
n_models, n_metrics = len(model_labels), len(metrics)

fig, axes = plt.subplots(n_metrics, n_models, figsize=(3.2 * n_models, 2.8 * n_metrics))

for col, lbl in enumerate(model_labels):
    color = MODEL_COLOR[lbl]
    dd = model_errors[lbl]
    for row, (metric, xlabel) in enumerate(metrics):
        ax = axes[row, col]
        x, y = dd[metric], dd["err"]
        ax.scatter(x, y, color=color, alpha=0.35, s=8, rasterized=True)

        coeffs = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                linestyle="--", label=f"slope={coeffs[0]:.3f}")
        ax.legend(fontsize=7, loc="upper left")

        if row == 0:
            ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        if row == n_metrics - 1:
            ax.set_xlabel(xlabel, fontsize=9)

plt.suptitle("Absolute error per molecule vs structural complexity (mol3d + fullerene combined, hp-tuned)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

### Per-molecule rank scatter

For every test molecule shared by all models, rank the models against each other by their MAE on that molecule (1 = best, last = worst). Ranking factors out molecules that are intrinsically hard for everyone, isolating relative skill. Y-axis is inverted (rank 1 at top); dashed line is the binned mean rank trend.

In [ ]:
# Molecules evaluated by every model, so ranks are directly comparable
common_keys = sorted(set.intersection(*[set(m.keys()) for m in err_maps.values()]))
print(f"Common test molecules across all {n_models} models: {len(common_keys)}")

err_mat = np.array([[err_maps[lbl][k] for lbl in model_labels] for k in common_keys])
rank_mat = np.vstack([rankdata(row, method="average") for row in err_mat])  # ties -> average rank

metric_arr = {m: np.array([get_metrics(*k)[i] for k in common_keys])
              for i, m in enumerate(["gd", "sd", "cn", "nr"])}

print("Overall mean rank (lower = better):")
for j, lbl in enumerate(model_labels):
    print(f"  {lbl:14s} {rank_mat[:, j].mean():.2f}")

In [ ]:
metrics_scatter = [
    ('gd', 'Graph diameter (bonds)', 'discrete'),
    ('sd', 'Spatial diameter (Å)', 'continuous'),
    ('cn', 'Cyclomatic number', 'discrete'),
    ('nr', 'Number of rings (SSSR)', 'discrete'),
]
rng = np.random.default_rng(42)

fig, axes = plt.subplots(len(metrics_scatter), n_models,
                         figsize=(3.2 * n_models, 2.8 * len(metrics_scatter)), squeeze=False)

for row, (metric, xlabel, kind) in enumerate(metrics_scatter):
    x_base = metric_arr[metric]
    for col, lbl in enumerate(model_labels):
        ax = axes[row, col]
        color = MODEL_COLOR[lbl]
        y_base = rank_mat[:, col]

        y_jitter = rng.uniform(-0.35, 0.35, size=len(y_base))
        if kind == "discrete":
            x_jitter = rng.uniform(-0.35, 0.35, size=len(x_base))
        else:
            x_range = x_base.max() - x_base.min()
            x_jitter = rng.uniform(-0.01 * x_range, 0.01 * x_range, size=len(x_base))

        ax.scatter(x_base + x_jitter, y_base + y_jitter, color=color, alpha=0.25, s=5,
                   rasterized=True, linewidths=0)

        stat, edges, _ = binned_statistic(x_base, y_base, statistic="mean", bins=12)
        centers = 0.5 * (edges[:-1] + edges[1:])
        valid = ~np.isnan(stat)
        ax.plot(centers[valid], stat[valid], color="black", linewidth=1.5, linestyle="--")

        ax.set_ylim(n_models + 0.6, 0.4)  # invert: rank 1 at top
        ax.set_yticks(range(1, n_models + 1))
        ax.tick_params(axis="both", labelsize=7)

        if row == 0:
            ax.set_title(lbl, fontsize=9, fontweight="bold", color=color)
        if col == 0:
            ax.set_ylabel(f"Rank\n({xlabel})", fontsize=8)
        if row == len(metrics_scatter) - 1:
            ax.set_xlabel(xlabel, fontsize=8)

plt.suptitle("Per-molecule rank scatter (mol3d + fullerene combined, hp-tuned, rank 1 = best)",
             fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## Structural Complexity Analysis — Split by Source

Same two analyses (error vs. structural complexity, per-molecule rank), but computed separately within each source dataset instead of pooling mol3d and fullerene molecules together. Useful since the two subsets occupy very different regions of structural-complexity space (see the bimodal x-axes in the combined plots above) — pooling them can hide a model doing well on one subset and poorly on the other.

In [ ]:
def build_source_data(source):
    data = {}
    for lbl in model_labels:
        gd_l, sd_l, cn_l, nr_l, err_l = [], [], [], [], []
        for (src, index), err in err_maps[lbl].items():
            if src != source:
                continue
            m = get_metrics(src, index)
            if m is None:
                continue
            gd, sd, cn, nr = m
            gd_l.append(gd); sd_l.append(sd); cn_l.append(cn); nr_l.append(nr); err_l.append(err)
        data[lbl] = {"gd": np.array(gd_l), "sd": np.array(sd_l),
                     "cn": np.array(cn_l), "nr": np.array(nr_l), "err": np.array(err_l)}
    return data

model_errors_mol3d = build_source_data("mol3d")
model_errors_fullerene = build_source_data("fullerene")

for src, data_src in [("mol3d", model_errors_mol3d), ("fullerene", model_errors_fullerene)]:
    print(src)
    for lbl in model_labels:
        print(f"  {lbl:14s} n={len(data_src[lbl]['err'])}")

### Absolute error per molecule vs structural complexity — by source

In [ ]:
def plot_error_vs_complexity(data_src, title_suffix):
    fig, axes = plt.subplots(n_metrics, n_models, figsize=(3.2 * n_models, 2.8 * n_metrics))
    for col, lbl in enumerate(model_labels):
        color = MODEL_COLOR[lbl]
        dd = data_src[lbl]
        for row, (metric, xlabel) in enumerate(metrics):
            ax = axes[row, col]
            x, y = dd[metric], dd["err"]
            ax.scatter(x, y, color=color, alpha=0.35, s=10, rasterized=True)

            if len(x) >= 2 and np.ptp(x) > 0:
                coeffs = np.polyfit(x, y, 1)
                x_line = np.linspace(x.min(), x.max(), 200)
                ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                        linestyle="--", label=f"slope={coeffs[0]:.3f}")
                ax.legend(fontsize=7, loc="upper left")

            if row == 0:
                ax.set_title(lbl, fontsize=10)
            if col == 0:
                ax.set_ylabel("Absolute error", fontsize=9)
            if row == n_metrics - 1:
                ax.set_xlabel(xlabel, fontsize=9)

    plt.suptitle(f"Absolute error per molecule vs structural complexity — {title_suffix} (hp-tuned)",
                 fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()

plot_error_vs_complexity(model_errors_mol3d, "mol3d only")

In [ ]:
plot_error_vs_complexity(model_errors_fullerene, "fullerene only")

### Per-molecule rank scatter — by source

Ranks are recomputed within each source subset (still 1–8 across the same 8 models, just ranked only against other molecules from that source).

In [ ]:
def plot_rank_scatter_by_source(source, title_suffix, seed=42):
    keys = [k for k in common_keys if k[0] == source]
    err_mat_src = np.array([[err_maps[lbl][k] for lbl in model_labels] for k in keys])
    rank_mat_src = np.vstack([rankdata(row, method="average") for row in err_mat_src])
    metric_arr_src = {m: np.array([get_metrics(*k)[i] for k in keys])
                       for i, m in enumerate(["gd", "sd", "cn", "nr"])}

    print(f"{title_suffix}: {len(keys)} common molecules. Mean rank (lower = better):")
    for j, lbl in enumerate(model_labels):
        print(f"  {lbl:14s} {rank_mat_src[:, j].mean():.2f}")

    rng_local = np.random.default_rng(seed)
    fig, axes = plt.subplots(len(metrics_scatter), n_models,
                             figsize=(3.2 * n_models, 2.8 * len(metrics_scatter)), squeeze=False)

    for row, (metric, xlabel, kind) in enumerate(metrics_scatter):
        x_base = metric_arr_src[metric]
        x_range = x_base.max() - x_base.min()
        for col, lbl in enumerate(model_labels):
            ax = axes[row, col]
            color = MODEL_COLOR[lbl]
            y_base = rank_mat_src[:, col]

            y_jitter = rng_local.uniform(-0.35, 0.35, size=len(y_base))
            if kind == "discrete":
                x_jitter = rng_local.uniform(-0.35, 0.35, size=len(x_base))
            elif x_range > 0:
                x_jitter = rng_local.uniform(-0.01 * x_range, 0.01 * x_range, size=len(x_base))
            else:
                x_jitter = np.zeros(len(x_base))

            ax.scatter(x_base + x_jitter, y_base + y_jitter, color=color, alpha=0.3, s=6,
                       rasterized=True, linewidths=0)

            if len(np.unique(x_base)) > 1:
                stat, edges, _ = binned_statistic(x_base, y_base, statistic="mean",
                                                   bins=min(12, len(np.unique(x_base))))
                centers = 0.5 * (edges[:-1] + edges[1:])
                valid = ~np.isnan(stat)
                ax.plot(centers[valid], stat[valid], color="black", linewidth=1.5, linestyle="--")

            ax.set_ylim(n_models + 0.6, 0.4)
            ax.set_yticks(range(1, n_models + 1))
            ax.tick_params(axis="both", labelsize=7)

            if row == 0:
                ax.set_title(lbl, fontsize=9, fontweight="bold", color=color)
            if col == 0:
                ax.set_ylabel(f"Rank\n({xlabel})", fontsize=8)
            if row == len(metrics_scatter) - 1:
                ax.set_xlabel(xlabel, fontsize=8)

    plt.suptitle(f"Per-molecule rank scatter — {title_suffix} (hp-tuned, rank 1 = best)", fontsize=13, y=1.01)
    plt.tight_layout()
    plt.show()
    return rank_mat_src

rank_mat_mol3d = plot_rank_scatter_by_source("mol3d", "mol3d only")

In [ ]:
rank_mat_fullerene = plot_rank_scatter_by_source("fullerene", "fullerene only")

## Fifth Metric — CT Hasse Graph Diameter

From `../mol3d/hasse_graph_diameter.ipynb`: the CT model message-passes over a 3-rank cell complex (atoms/bonds/rings) using 6 neighborhood matrices (`adj00`, `icd01`, `adj11`, `icd02`, `icd12`, `adj22`). Building one combined graph over all `n_atoms + n_bonds + n_rings` cells from those 6 relations and taking its diameter gives a CT-specific structural-complexity metric the 4 atom-only metrics above can't see.

- mol3d values are reused directly from `../mol3d/hasse_graph_per_molecule.npz` — same split file (`mol3d/data_split.json`), same global index, no recomputation needed.
- fullerene values were computed fresh for this notebook (`fullerene_hasse_per_molecule.npz`), using the identical `hasse_graph_diameter()` function applied to `make_matrices()` output from `fullerene_loader._iter_fullerene_mols`.

(As with the other structural metrics, this is a molecule-only property computed once — identical regardless of hyperparameters.)

In [ ]:
mol3d_hasse_npz = np.load("../mol3d/hasse_graph_per_molecule.npz")
mol3d_hasse_pos = {int(idx): i for i, idx in enumerate(mol3d_hasse_npz["index"])}
mol3d_hd = mol3d_hasse_npz["hasse_diameter"]

fullerene_hasse_npz = np.load("fullerene_hasse_per_molecule.npz")
fullerene_hasse_pos = {int(idx): i for i, idx in enumerate(fullerene_hasse_npz["index"])}
fullerene_hd = fullerene_hasse_npz["hasse_diameter"]

def get_hasse(source, index):
    if source == "mol3d":
        pos = mol3d_hasse_pos.get(index)
        return float(mol3d_hd[pos]) if pos is not None else None
    pos = fullerene_hasse_pos.get(index)
    return float(fullerene_hd[pos]) if pos is not None else None

print(f"mol3d hasse diameter:     n={len(mol3d_hd):,}  "
      f"min={mol3d_hd.min()} max={mol3d_hd.max()} mean={mol3d_hd.mean():.2f}")
print(f"fullerene hasse diameter: n={len(fullerene_hd):,}  "
      f"min={fullerene_hd.min()} max={fullerene_hd.max()} mean={fullerene_hd.mean():.2f}")

### Hasse diameter distribution by source

Note this is over the *full* mol3d (10k) and fullerene (7,100) pools, not just the test molecules — same scope as the equivalent histogram in `../mol3d/hasse_graph_diameter.ipynb`.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
all_vals = np.arange(min(mol3d_hd.min(), fullerene_hd.min()), max(mol3d_hd.max(), fullerene_hd.max()) + 1)
for arr, lbl, color in [(mol3d_hd, f"mol3d (n={len(mol3d_hd):,})", FAMILY_COLOR["CT"]),
                         (fullerene_hd, f"fullerene (n={len(fullerene_hd):,})", FAMILY_COLOR["GAT"])]:
    counts = np.array([(arr == v).sum() for v in all_vals])
    ax.bar(all_vals, counts / len(arr) * 100, alpha=0.6, label=lbl, color=color,
           edgecolor="white", linewidth=0.3)
ax.set_xlabel("Hasse graph diameter (atom+bond+ring cells)")
ax.set_ylabel("% of molecules")
ax.set_title("Hasse graph diameter — mol3d vs fullerene")
ax.legend()
plt.tight_layout()
plt.show()

### Hasse vs atom-only graph diameter — max/average difference

Computed separately per source (mol3d, fullerene), over each source's test molecules, since the two populations occupy very different regions of this metric.

In [ ]:
def hasse_vs_atom_diff(source):
    keys = [k for k in common_keys if k[0] == source]
    hd = np.array([get_hasse(*k) for k in keys])
    gd = np.array([get_metrics(*k)[0] for k in keys])  # atom-only graph diameter
    diff = hd - gd
    abs_diff = np.abs(diff)
    worst = keys[int(np.argmax(abs_diff))]
    print(f"{source} (n={len(keys)}):")
    print(f"  Average difference (hasse - atom):  {diff.mean():+.3f}")
    print(f"  Average absolute difference:        {abs_diff.mean():.3f}")
    print(f"  Maximum absolute difference:        {abs_diff.max():.0f}  (molecule index {worst[1]})")
    print(f"  Max/min signed difference:           {diff.max():+.0f} / {diff.min():+.0f}")
    print(f"  Correlation(hasse, atom):            {np.corrcoef(hd, gd)[0,1]:.3f}")
    return hd, gd, diff

hd_mol3d, gd_mol3d, diff_mol3d = hasse_vs_atom_diff("mol3d")
print()
hd_fullerene, gd_fullerene, diff_fullerene = hasse_vs_atom_diff("fullerene")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
rng = np.random.default_rng(0)
for ax, gd, hd, title, color in [
    (axes[0], gd_mol3d, hd_mol3d, "mol3d", FAMILY_COLOR["CT"]),
    (axes[1], gd_fullerene, hd_fullerene, "fullerene", FAMILY_COLOR["GAT"]),
]:
    jitter = rng.uniform(-0.3, 0.3, size=(len(gd), 2))
    ax.scatter(gd + jitter[:, 0], hd + jitter[:, 1], s=8, alpha=0.15, color=color, rasterized=True)
    lims = [min(gd.min(), hd.min()) - 1, max(gd.max(), hd.max()) + 1]
    ax.plot(lims, lims, color="gray", linestyle=":", linewidth=1, label="y = x")
    ax.set_xlabel("Atom-only graph diameter (bonds)")
    ax.set_ylabel("Hasse graph diameter (cells)")
    ax.set_title(title)
    ax.legend()
plt.suptitle("Hasse vs atom-only diameter, by source", y=1.02)
plt.tight_layout()
plt.show()

### Model error vs Hasse graph diameter

Does the Hasse diameter explain error better than the atom-only graph diameter for any model? Shown for all models (not just CT) — GNNs/CIN don't use rings the same way, but the Hasse diameter is still a property of the molecule and might still correlate with difficulty (e.g. as a proxy for molecule size/compactness).

In [ ]:
def plot_err_vs_hasse(keys, title):
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        x = np.array([get_hasse(*k) for k in keys])
        y = np.array([err_maps[lbl][k] for k in keys])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=7, loc="upper left")
        ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        ax.set_xlabel("Hasse graph diameter", fontsize=9)
    plt.suptitle(f"Absolute error vs Hasse graph diameter — {title} (hp-tuned)", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_hasse(common_keys, "mol3d + fullerene combined")

In [ ]:
plot_err_vs_hasse([k for k in common_keys if k[0] == "mol3d"], "mol3d only")

In [ ]:
plot_err_vs_hasse([k for k in common_keys if k[0] == "fullerene"], "fullerene only")

### Model rank vs Hasse graph diameter

In [ ]:
def plot_rank_vs_hasse(keys, rank_mat_local, title, seed=42):
    x_base = np.array([get_hasse(*k) for k in keys])
    rng_local = np.random.default_rng(seed)
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.4), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        y_base = rank_mat_local[:, col]
        y_jitter = rng_local.uniform(-0.35, 0.35, size=len(y_base))
        x_jitter = rng_local.uniform(-0.35, 0.35, size=len(x_base))
        ax.scatter(x_base + x_jitter, y_base + y_jitter, color=color, alpha=0.3, s=6,
                   rasterized=True, linewidths=0)
        if len(np.unique(x_base)) > 1:
            stat, edges, _ = binned_statistic(x_base, y_base, statistic="mean",
                                               bins=min(12, len(np.unique(x_base))))
            centers = 0.5 * (edges[:-1] + edges[1:])
            valid = ~np.isnan(stat)
            ax.plot(centers[valid], stat[valid], color="black", linewidth=1.5, linestyle="--")
        ax.set_ylim(n_models + 0.6, 0.4)
        ax.set_yticks(range(1, n_models + 1))
        ax.tick_params(axis="both", labelsize=7)
        ax.set_title(lbl, fontsize=9, fontweight="bold", color=color)
        if col == 0:
            ax.set_ylabel("Rank", fontsize=9)
        ax.set_xlabel("Hasse graph diameter", fontsize=8)
    plt.suptitle(f"Per-molecule rank vs Hasse graph diameter — {title} (hp-tuned, rank 1 = best)",
                 fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_rank_vs_hasse(common_keys, rank_mat, "mol3d + fullerene combined")

In [ ]:
plot_rank_vs_hasse([k for k in common_keys if k[0] == "mol3d"], rank_mat_mol3d, "mol3d only")

In [ ]:
plot_rank_vs_hasse([k for k in common_keys if k[0] == "fullerene"], rank_mat_fullerene, "fullerene only")

### MAE by Hasse diameter threshold (≤6 vs >6)

Splits each source's test molecules at Hasse diameter 6 and compares mean MAE. **Caveat for the combined view**: this split is heavily confounded with source — mol3d's Hasse diameter range is 3–21 (mostly >6) and fullerene's is 4–8 (splits near the middle), so "≤6" skews fullerene-heavy and ">6" skews mol3d-heavy. The mol3d-only and fullerene-only versions below isolate the diameter effect within a single source, though note the within-source group sizes are themselves imbalanced (most mol3d test molecules have diameter >6; most fullerene test molecules have diameter ≤6).

In [ ]:
def plot_mae_by_hasse_threshold(keys, title, threshold=6):
    low_keys  = [k for k in keys if get_hasse(*k) <= threshold]
    high_keys = [k for k in keys if get_hasse(*k) > threshold]

    rows = []
    for lbl in model_labels:
        low_errs  = [err_maps[lbl][k] for k in low_keys]
        high_errs = [err_maps[lbl][k] for k in high_keys]
        rows.append({
            "label": lbl,
            "mae_low": np.mean(low_errs), "std_low": np.std(low_errs),
            "mae_high": np.mean(high_errs), "std_high": np.std(high_errs),
        })
    df_t = pd.DataFrame(rows).set_index("label").loc[model_labels]

    x = np.arange(len(df_t))
    width = 0.35
    colors = [MODEL_COLOR[lbl] for lbl in df_t.index]

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(x - width / 2, df_t["mae_low"], width, yerr=df_t["std_low"], color=colors,
           edgecolor="black", linewidth=0.6, capsize=3, label=f"Hasse ≤ {threshold}")
    ax.bar(x + width / 2, df_t["mae_high"], width, yerr=df_t["std_high"], color=colors, alpha=0.45,
           edgecolor="black", linewidth=0.6, capsize=3, label=f"Hasse > {threshold}")

    for xi, row in zip(x, df_t.itertuples()):
        ax.text(xi - width / 2, row.mae_low + row.std_low + 0.01, f"{row.mae_low:.3f}",
                ha="center", va="bottom", fontsize=7)
        ax.text(xi + width / 2, row.mae_high + row.std_high + 0.01, f"{row.mae_high:.3f}",
                ha="center", va="bottom", fontsize=7)

    ax.set_xticks(x)
    ax.set_xticklabels(df_t.index, rotation=30, ha="right")
    ax.set_ylabel("Test MAE")
    ax.set_title(f"MAE by Hasse diameter threshold — {title} (hp-tuned)\n"
                 f"(n≤{threshold}={len(low_keys)}, n>{threshold}={len(high_keys)})")
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.yaxis.grid(True, alpha=0.25)
    ax.set_axisbelow(True)
    ax.legend(title="Hasse diameter", loc="upper left", bbox_to_anchor=(1.01, 1), frameon=False)

    plt.tight_layout()
    plt.show()
    return df_t

_ = plot_mae_by_hasse_threshold(common_keys, "mol3d + fullerene combined")

In [ ]:
_ = plot_mae_by_hasse_threshold([k for k in common_keys if k[0] == "mol3d"], "mol3d only")

In [ ]:
_ = plot_mae_by_hasse_threshold([k for k in common_keys if k[0] == "fullerene"], "fullerene only")

### Error vs Hasse-minus-atom diameter, and vs Hasse/atom diameter ratio

Two more views of the same relationship, over every test molecule (mol3d + fullerene combined): the signed difference `hasse − atom` (how much the ring/bond shortcuts shrink or grow the diameter in absolute terms) and the ratio `hasse / atom` (the same effect in relative/multiplicative terms — more comparable across molecules of very different size).

In [ ]:
def plot_err_vs_hasse_diff(keys, title):
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        hasse = np.array([get_hasse(*k) for k in keys])
        atom = np.array([get_metrics(*k)[0] for k in keys])
        x = hasse - atom
        y = np.array([err_maps[lbl][k] for k in keys])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=7, loc="upper left")
        ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        ax.set_xlabel("Hasse − atom diameter", fontsize=9)
    plt.suptitle(f"Absolute error vs (Hasse − atom) graph diameter — {title} (hp-tuned)", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_hasse_diff(common_keys, "mol3d + fullerene combined")

In [ ]:
def plot_err_vs_hasse_ratio(keys, title):
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        hasse = np.array([get_hasse(*k) for k in keys])
        atom = np.array([get_metrics(*k)[0] for k in keys])
        valid_mask = atom > 0  # guard against division by zero (shouldn't occur, but be safe)
        x = hasse[valid_mask] / atom[valid_mask]
        y = np.array([err_maps[lbl][k] for k, keep in zip(keys, valid_mask) if keep])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=7, loc="upper left")
        ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        ax.set_xlabel("Hasse / atom diameter", fontsize=9)
    plt.suptitle(f"Absolute error vs (Hasse / atom) graph diameter ratio — {title} (hp-tuned)", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_hasse_ratio(common_keys, "mol3d + fullerene combined")

## Sixth Metric — Atom-Restricted Hasse Diameter

The previous "Hasse graph diameter" measures the longest shortest path between *any two cells* (atom, bond, or ring) — which is how a stray peripheral bond/ring cell could push it slightly above the atom-only graph diameter (ratio > 1), as seen above.

This metric fixes that: it's still the longest shortest path over the same combined atom+bond+ring graph (all 6 CT neighborhoods — `adj00`, `icd01`, `adj11`, `icd02`, `icd12`, `adj22` — are still fair game *as intermediate hops*), but the two **endpoints are restricted to atoms**. This is exactly "the longest shortest path between two atoms, using every way the CT model can move between cells" — the path can duck through a bond or ring cell as a shortcut, it just has to start and end on atoms.

By construction this can never exceed the plain atom-only graph diameter (same endpoints, strictly more edges available to route through) — confirmed empirically: strictly smaller for all 7,100 fullerenes, zero ties.

Scope: fullerene computed for the full 7,100-molecule pool (`fullerene_atom_hasse_per_molecule.npz`); mol3d computed for the 2,000 test-set molecules only (`mol3d_atom_hasse_test_only.npz`) rather than the full 10k pool, since a second ~15-minute `Mol3dCT` reload isn't needed for anything downstream — every consumer of this metric is a test-set analysis anyway.

In [ ]:
mol3d_atom_hasse_npz = np.load("mol3d_atom_hasse_test_only.npz")
mol3d_atom_hasse_pos = {int(idx): i for i, idx in enumerate(mol3d_atom_hasse_npz["index"])}
mol3d_atom_hd = mol3d_atom_hasse_npz["atom_hasse_diameter"]

fullerene_atom_hasse_npz = np.load("fullerene_atom_hasse_per_molecule.npz")
fullerene_atom_hasse_pos = {int(idx): i for i, idx in enumerate(fullerene_atom_hasse_npz["index"])}
fullerene_atom_hd = fullerene_atom_hasse_npz["atom_hasse_diameter"]

def get_atom_hasse(source, index):
    if source == "mol3d":
        pos = mol3d_atom_hasse_pos.get(index)
        return float(mol3d_atom_hd[pos]) if pos is not None else None
    pos = fullerene_atom_hasse_pos.get(index)
    return float(fullerene_atom_hd[pos]) if pos is not None else None

print(f"mol3d atom-hasse (test only): n={len(mol3d_atom_hd):,}  "
      f"min={mol3d_atom_hd.min()} max={mol3d_atom_hd.max()} mean={mol3d_atom_hd.mean():.2f}")
print(f"fullerene atom-hasse:         n={len(fullerene_atom_hd):,}  "
      f"min={fullerene_atom_hd.min()} max={fullerene_atom_hd.max()} mean={fullerene_atom_hd.mean():.2f}")

### Atom-restricted Hasse diameter vs atom-only graph diameter — max/average difference

In [ ]:
def atom_hasse_vs_atom_diff(source):
    keys = [k for k in common_keys if k[0] == source]
    ah = np.array([get_atom_hasse(*k) for k in keys])
    gd = np.array([get_metrics(*k)[0] for k in keys])
    diff = ah - gd
    abs_diff = np.abs(diff)
    worst = keys[int(np.argmax(abs_diff))]
    print(f"{source} (n={len(keys)}):")
    print(f"  Average difference (atom_hasse - atom):  {diff.mean():+.3f}")
    print(f"  Average absolute difference:             {abs_diff.mean():.3f}")
    print(f"  Maximum absolute difference:             {abs_diff.max():.0f}  (molecule index {worst[1]})")
    print(f"  Max/min signed difference:                {diff.max():+.0f} / {diff.min():+.0f}")
    print(f"  Correlation(atom_hasse, atom):            {np.corrcoef(ah, gd)[0,1]:.3f}")
    return ah, gd, diff

ah_mol3d, gd_mol3d_2, diff_ah_mol3d = atom_hasse_vs_atom_diff("mol3d")
print()
ah_fullerene, gd_fullerene_2, diff_ah_fullerene = atom_hasse_vs_atom_diff("fullerene")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
rng = np.random.default_rng(0)
for ax, gd, ah, title, color in [
    (axes[0], gd_mol3d_2, ah_mol3d, "mol3d", FAMILY_COLOR["CT"]),
    (axes[1], gd_fullerene_2, ah_fullerene, "fullerene", FAMILY_COLOR["GAT"]),
]:
    jitter = rng.uniform(-0.3, 0.3, size=(len(gd), 2))
    ax.scatter(gd + jitter[:, 0], ah + jitter[:, 1], s=8, alpha=0.15, color=color, rasterized=True)
    lims = [min(gd.min(), ah.min()) - 1, max(gd.max(), ah.max()) + 1]
    ax.plot(lims, lims, color="gray", linestyle=":", linewidth=1, label="y = x")
    ax.set_xlabel("Atom-only graph diameter (bonds)")
    ax.set_ylabel("Atom-restricted Hasse diameter")
    ax.set_title(title)
    ax.legend()
plt.suptitle("Atom-restricted Hasse vs atom-only diameter, by source", y=1.02)
plt.tight_layout()
plt.show()

### Model error vs (atom-restricted Hasse − atom) diameter, and vs the ratio

Same two views as before, but with the confound-free metric — this time the ratio is guaranteed ≤ 1.0 for every molecule.

In [ ]:
def plot_err_vs_atom_hasse_diff(keys, title):
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        ah = np.array([get_atom_hasse(*k) for k in keys])
        atom = np.array([get_metrics(*k)[0] for k in keys])
        x = ah - atom
        y = np.array([err_maps[lbl][k] for k in keys])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=7, loc="upper left")
        ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        ax.set_xlabel("Atom-restricted Hasse − atom diameter", fontsize=9)
    plt.suptitle(f"Absolute error vs (atom-restricted Hasse − atom) diameter — {title} (hp-tuned)", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_atom_hasse_diff(common_keys, "mol3d + fullerene combined")

In [ ]:
def plot_err_vs_atom_hasse_ratio(keys, title):
    fig, axes = plt.subplots(1, n_models, figsize=(3.2 * n_models, 3.2), squeeze=False)
    axes = axes[0]
    for col, lbl in enumerate(model_labels):
        ax = axes[col]
        color = MODEL_COLOR[lbl]
        ah = np.array([get_atom_hasse(*k) for k in keys])
        atom = np.array([get_metrics(*k)[0] for k in keys])
        valid_mask = atom > 0
        x = ah[valid_mask] / atom[valid_mask]
        y = np.array([err_maps[lbl][k] for k, keep in zip(keys, valid_mask) if keep])
        ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=7, loc="upper left")
        ax.set_title(lbl, fontsize=10)
        if col == 0:
            ax.set_ylabel("Absolute error", fontsize=9)
        ax.set_xlabel("Atom-restricted Hasse / atom diameter", fontsize=9)
    plt.suptitle(f"Absolute error vs (atom-restricted Hasse / atom) diameter ratio — {title} (hp-tuned)", fontsize=13, y=1.05)
    plt.tight_layout()
    plt.show()

plot_err_vs_atom_hasse_ratio(common_keys, "mol3d + fullerene combined")

## CIN / CIN++ — Model-Specific Hasse Graph Diameter

The Hasse/atom-restricted-Hasse metrics above are built from CT's 6 neighborhoods (`adj00`/`icd01`/`adj11`/`icd02`/`icd12`/`adj22`). CIN and CIN++ message-pass over a *different* set of neighborhoods (`hasse_graph_diameter_cin.py`, mirroring `models/cin.py::CINLayer.forward`): CIN uses `up0`/`up1`/`boundary1`/`boundary2` only, CIN++ adds `down1`/`down2`, and neither ever has an atom↔ring relation the way CT's `icd02` does. So each variant gets its own Hasse graph diameter here, computed directly from the index tensors each model actually trains on -- mol3d side reused from `../mol3d/hasse_graph_cin_per_molecule.npz`, fullerene side from `fullerene_hasse_cin_per_molecule.npz` (both test split only).

In [ ]:
mol3d_cin_hasse_npz = np.load("../mol3d/hasse_graph_cin_per_molecule.npz")
mol3d_cin_hasse_pos = {int(idx): i for i, idx in enumerate(mol3d_cin_hasse_npz["index"])}
mol3d_cin_hd = mol3d_cin_hasse_npz["cin_hasse_diameter"]
mol3d_cinpp_hd = mol3d_cin_hasse_npz["cinpp_hasse_diameter"]

fullerene_cin_hasse_npz = np.load("fullerene_hasse_cin_per_molecule.npz")
fullerene_cin_hasse_pos = {int(idx): i for i, idx in enumerate(fullerene_cin_hasse_npz["index"])}
fullerene_cin_hd = fullerene_cin_hasse_npz["cin_hasse_diameter"]
fullerene_cinpp_hd = fullerene_cin_hasse_npz["cinpp_hasse_diameter"]

def get_cin_hasse(source, index, variant):
    if source == "mol3d":
        pos = mol3d_cin_hasse_pos.get(index)
        if pos is None:
            return None
        return float(mol3d_cin_hd[pos]) if variant == "CIN" else float(mol3d_cinpp_hd[pos])
    pos = fullerene_cin_hasse_pos.get(index)
    if pos is None:
        return None
    return float(fullerene_cin_hd[pos]) if variant == "CIN" else float(fullerene_cinpp_hd[pos])

print(f"mol3d     CIN   hasse diameter: n={len(mol3d_cin_hd):,}  min={mol3d_cin_hd.min()} max={mol3d_cin_hd.max()} mean={mol3d_cin_hd.mean():.2f}")
print(f"mol3d     CINpp hasse diameter: n={len(mol3d_cinpp_hd):,}  min={mol3d_cinpp_hd.min()} max={mol3d_cinpp_hd.max()} mean={mol3d_cinpp_hd.mean():.2f}")
print(f"fullerene CIN   hasse diameter: n={len(fullerene_cin_hd):,}  min={fullerene_cin_hd.min()} max={fullerene_cin_hd.max()} mean={fullerene_cin_hd.mean():.2f}")
print(f"fullerene CINpp hasse diameter: n={len(fullerene_cinpp_hd):,}  min={fullerene_cinpp_hd.min()} max={fullerene_cinpp_hd.max()} mean={fullerene_cinpp_hd.mean():.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4.5))
for ax, label, variant in [(axes[0], "CIN", "CIN"), (axes[1], "CINpp", "CINpp")]:
    if label not in err_maps:
        ax.set_visible(False)
        continue
    err_by_key = err_maps[label]
    keys = [k for k in err_by_key if get_cin_hasse(*k, variant) is not None]
    x = np.array([get_cin_hasse(*k, variant) for k in keys])
    y = np.array([err_by_key[k] for k in keys])
    color = MODEL_COLOR[label]
    ax.scatter(x, y, color=color, alpha=0.3, s=8, rasterized=True)
    if len(x) >= 2 and np.ptp(x) > 0:
        coeffs = np.polyfit(x, y, 1)
        x_line = np.linspace(x.min(), x.max(), 200)
        ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.8,
                linestyle="--", label=f"slope={coeffs[0]:.4f}")
        ax.legend(fontsize=8, loc="upper left")
    ax.set_title(f"{label} (own Hasse graph)", fontsize=10)
    ax.set_xlabel(f"{label} Hasse graph diameter")
    ax.set_ylabel("Absolute error")

plt.suptitle("mol3d_fullerene — CIN/CINpp test error vs their own (model-specific) Hasse graph diameter (combined)",
             fontsize=12, y=1.03)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(2, n_models, figsize=(3.0 * n_models, 5.6), squeeze=False)

for row, (variant, hd_name) in enumerate([("CIN", "CIN Hasse diameter"), ("CINpp", "CINpp Hasse diameter")]):
    for col, lbl in enumerate(model_labels):
        ax = axes[row, col]
        err_by_key = err_maps[lbl]
        keys = [k for k in err_by_key if get_cin_hasse(*k, variant) is not None]
        x = np.array([get_cin_hasse(*k, variant) for k in keys])
        y = np.array([err_by_key[k] for k in keys])
        color = MODEL_COLOR[lbl]
        ax.scatter(x, y, color=color, alpha=0.3, s=6, rasterized=True)
        if len(x) >= 2 and np.ptp(x) > 0:
            coeffs = np.polyfit(x, y, 1)
            x_line = np.linspace(x.min(), x.max(), 200)
            ax.plot(x_line, np.polyval(coeffs, x_line), color="black", linewidth=1.5,
                    linestyle="--", label=f"slope={coeffs[0]:.3f}")
            ax.legend(fontsize=6, loc="upper left")
        if row == 0:
            ax.set_title(lbl, fontsize=9)
        if col == 0:
            ax.set_ylabel(f"Absolute error\n({hd_name})", fontsize=8)
        if row == 1:
            ax.set_xlabel(hd_name, fontsize=8)

plt.suptitle("mol3d_fullerene — all models' test error vs CIN / CINpp Hasse graph diameter (combined)",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.show()